# Fraud Detection Project

## Objective
To build a machine learning model that detects fraudulent financial
transactions using historical transaction data.

## Libraries Used
Pandas and NumPy are used for data analysis and numerical operations.



In [1]:
import pandas as pd
import numpy as np 


## Data Loading and Preview

The transaction dataset is loaded from a CSV file into a Pandas DataFrame.
The first few rows are displayed to understand the structure, columns,
and type of data available for fraud detection.

In [2]:
df = pd.read_csv("Fraud.csv")
df.head()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## Dataset Size

This step is used to understand the size of the dataset.
It shows the total number of rows (transactions) and columns (features),
which helps assess the scale of data and computational requirements.


In [3]:
df.shape


(6362620, 11)

## Missing Values Analysis

Checking for missing values is an important data cleaning step.
Missing values can negatively impact model performance and may require
imputation or removal. This step helps identify whether such treatment
is necessary.


In [4]:
df.isnull().sum()


step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

## Encoding Categorical Variables

Machine learning models require numerical input.
The transaction type feature is categorical in nature, so it is converted
into numerical form using Label Encoding. This allows the model to process
the transaction type effectively.


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])

df['type'].head()


0    3
1    3
2    4
3    1
4    3
Name: type, dtype: int64

## Statistical Summary of Numerical Features

This step provides a statistical summary of key numerical variables such as
transaction amount and account balances. It helps in understanding the data
distribution, detecting extreme values, and identifying potential outliers
that may indicate fraudulent behavior.


In [6]:
df[['amount', 'oldbalanceOrg', 'newbalanceOrig']].describe()


,amount,oldbalanceOrg,newbalanceOrig
count,6.362620e+06,6.362620e+06,6.362620e+06
mean,1.798619e+05,8.338831e+05,8.551137e+05
std,6.038582e+05,2.888243e+06,2.924049e+06
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.338957e+04,0.000000e+00,0.000000e+00
50%,7.487194e+04,1.420800e+04,0.000000e+00
75%,2.087215e+05,1.073152e+05,1.442584e+05
max,9.244552e+07,5.958504e+07,4.958504e+07


## Correlation Analysis

Correlation analysis is performed to identify relationships between numerical
features. Highly correlated variables may indicate multi-collinearity, which
can affect model interpretability. This step helps in understanding dependency
between account balance features.


In [7]:
df[['oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest']].corr()


,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
oldbalanceOrg,1.000000,0.998803,0.066243,0.042029
newbalanceOrig,0.998803,1.000000,0.067812,0.041837
oldbalanceDest,0.066243,0.067812,1.000000,0.976569
newbalanceDest,0.042029,0.041837,0.976569,1.000000


## Feature Selection and Target Variable

The dataset is divided into input features (X) and the target variable (y).
The target variable `isFraud` represents whether a transaction is fraudulent.
Identifier columns such as `nameOrig` and `nameDest` are removed because they
do not provide predictive value and may introduce noise into the model.


In [8]:
X = df.drop(['isFraud', 'nameOrig', 'nameDest'], axis=1)
y = df['isFraud']

X.shape, y.shape


((6362620, 8), (6362620,))

## Train–Test Data Split

The dataset is split into training (calibration) and testing (validation) sets.
This allows the model to be trained on historical data and evaluated on unseen
data to assess its generalization performance. Stratified sampling is used to
maintain the original fraud class distribution.


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Split completed")


Split completed


## Random Forest Model Training

A Random Forest classifier is used to detect fraudulent transactions.  
Random Forest is chosen because it:

- Handles large datasets efficiently
- Captures non-linear relationships
- Is robust to outliers and correlated features
- Works well with imbalanced data (like fraud datasets)

The model is trained on the calibration (training) dataset using 50 trees
and a maximum depth of 10 to balance accuracy and computational cost.


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print("Model trained")


Model trained


## Model Evaluation

The trained Random Forest model is evaluated on the validation (test) dataset
using multiple performance metrics:

- **Confusion Matrix:** Shows correct and incorrect predictions for both
  fraudulent and non-fraudulent transactions.
- **Classification Report:** Includes precision, recall, F1-score for each class.
  Recall is especially important for fraud detection, as missing a fraud
  transaction is costly.
- **ROC-AUC Score:** Measures the ability of the model to distinguish between
  fraud and non-fraud transactions. A value closer to 1 indicates excellent
  performance.


In [11]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

y_pred = rf_model.predict(X_test)

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

roc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1])
print("ROC AUC:", roc)


Confusion Matrix
[[1906317       5]
 [    731    1733]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906322
           1       1.00      0.70      0.82      2464

    accuracy                           1.00   1908786
   macro avg       1.00      0.85      0.91   1908786
weighted avg       1.00      1.00      1.00   1908786

ROC AUC: 0.996730276043259


## Feature Importance

Feature importance indicates which variables contribute most to predicting
fraudulent transactions. The Random Forest model calculates how much each
feature reduces impurity across all trees. 

Higher importance means the feature has a stronger influence on fraud prediction.
This helps understand the key drivers of fraud and provides actionable insights
for the company.


In [12]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)


newbalanceDest    0.323165
oldbalanceOrg     0.286274
amount            0.130771
step              0.085142
type              0.082576
oldbalanceDest    0.060013
newbalanceOrig    0.030621
isFlaggedFraud    0.001437
dtype: float64

## Key Fraud Predictors

- **newbalanceDest**: Most important — sudden changes in the destination balance often indicate fraud.
- **oldbalanceOrg**: High impact — unusual origin account balances can signal suspicious activity.
- **amount**: Large transaction amounts are more likely to be fraudulent.
- **step & type**: Timing and transaction type also influence fraud likelihood.
- **isFlaggedFraud**: Very low importance, likely because the model learns patterns from raw balances and amounts.


### Q7: What kind of prevention should be adopted while company updates its infrastructure?

To minimize fraudulent transactions while updating company infrastructure:

- Implement **real-time transaction monitoring** to detect suspicious activity immediately.
- Introduce **multi-factor authentication** and alerts for high-risk or unusual transactions.
- Continuously **update machine learning models** with new transaction data to adapt to evolving fraud patterns.
- Apply **threshold-based or anomaly detection systems** for unusual amounts or rapid balance changes.
- Ensure **secure storage and encryption** of all transaction and customer data to prevent tampering or leakage.

---

### Q8: Assuming these actions have been implemented, how would you determine if they work?

The effectiveness of implemented actions can be evaluated by:

- Tracking key metrics such as:
  - **Reduction in fraudulent transactions** compared to baseline
  - **False positive and false negative rates** on new transactions
  - **ROC-AUC or precision-recall**


## Conclusion

The Random Forest model successfully detected fraudulent transactions with high performance,
achieving a ROC-AUC of 0.99. Key predictors such as `newbalanceDest`, `oldbalanceOrg`, and
`amount` were identified, providing actionable insights for the company. 

The analysis demonstrates that careful data cleaning, feature selection, and model evaluation
can effectively support proactive fraud detection. Implementing real-time monitoring, multi-factor
authentication, and periodic audits will further enhance fraud prevention, while ongoing
model updates will ensure adaptability to evolving fraud patterns.

Overall, this project illustrates the application of data-driven methods to identify and
mitigate fraudulent transactions, supporting informed business decision-making and risk reduction.
